# Grouped Error and Fairness Diagnostics Lab


## Purpose

This lab extends model validation into grouped diagnostics.

The goal is not to claim that simple metrics solve fairness. The goal is to show how aggregate performance can hide uneven error burdens.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42

In [ ]:
X, y = make_classification(
    n_samples=6000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.65, 0.35],
    random_state=RANDOM_SEED,
)

frame = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
frame["target"] = y

rng = np.random.default_rng(RANDOM_SEED)
frame["group"] = rng.choice(["A", "B", "C"], size=len(frame), p=[0.50, 0.30, 0.20])

In [ ]:
features = [c for c in frame.columns if c.startswith("feature_")]

X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    frame[features],
    frame["target"],
    frame["group"],
    test_size=0.30,
    stratify=frame["target"],
    random_state=RANDOM_SEED,
)

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)

model.fit(X_train, y_train)

audit = X_test.copy()
audit["target"] = y_test.to_numpy()
audit["group"] = group_test.to_numpy()
audit["score"] = model.predict_proba(X_test)[:, 1]
audit["prediction"] = (audit["score"] >= 0.50).astype(int)

## Grouped Diagnostics Function

This function computes selection rate, false positive rate, and false negative rate by group.

In [ ]:
def grouped_diagnostics(data: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for group_name, group_frame in data.groupby("group"):
        target = group_frame["target"]
        prediction = group_frame["prediction"]

        tp = ((target == 1) & (prediction == 1)).sum()
        tn = ((target == 0) & (prediction == 0)).sum()
        fp = ((target == 0) & (prediction == 1)).sum()
        fn = ((target == 1) & (prediction == 0)).sum()

        rows.append(
            {
                "group": group_name,
                "n": len(group_frame),
                "base_rate": target.mean(),
                "selection_rate": prediction.mean(),
                "false_positive_rate": fp / max(fp + tn, 1),
                "false_negative_rate": fn / max(fn + tp, 1),
            }
        )

    return pd.DataFrame(rows)

grouped_diagnostics(audit)

## Threshold Sensitivity

Fairness and reliability are often threshold-sensitive. A single threshold can shift who receives false positives or false negatives.

In [ ]:
def diagnostics_at_threshold(data: pd.DataFrame, threshold: float) -> pd.DataFrame:
    temp = data.copy()
    temp["prediction"] = (temp["score"] >= threshold).astype(int)
    result = grouped_diagnostics(temp)
    result["threshold"] = threshold
    return result

threshold_results = pd.concat(
    [diagnostics_at_threshold(audit, t) for t in np.linspace(0.20, 0.80, 13)],
    ignore_index=True,
)

threshold_results.head()

In [ ]:
threshold_results.to_csv("../outputs/threshold_grouped_diagnostics.csv", index=False)
threshold_results.tail()

## Interpretation Checklist

- Which group has the highest false positive rate?
- Which group has the highest false negative rate?
- Does the threshold reduce one harm while increasing another?
- Are the group labels legally and ethically appropriate to use?
- Who should review the results before deployment?